# 🏫 Teacher Workspace Agent — GRPO Training

Trains **Llama-3.2-1B-Instruct** (4-bit LoRA) with GRPO on the Teacher Workspace Environment.

The agent learns to orchestrate Google Workspace tools to complete teacher admin tasks.

**Runtime:** T4 GPU | **Est. time:** ~45-60 min for 200 steps

## 1. Install Dependencies

In [ ]:
!pip install unsloth==2024.12.4 -q
!pip install trl==0.12.2 -q
!pip install openenv-core[core]>=0.2.2 -q
!pip install wandb -q
!pip install matplotlib -q
!pip install huggingface_hub -q

## 2. Clone Environment

In [ ]:
!git clone https://github.com/amydosomething/teacher-workspace.git
%cd teacher-workspace
!pip install -e . -q

## 3. Imports & Setup

In [ ]:
import sys, os, json, re, random, torch, time
import matplotlib.pyplot as plt
from typing import List, Dict, Any

sys.path.insert(0, '/content/teacher-workspace')

from server.teacher_workspace_env_environment import TeacherWorkspaceEnvironment
from models import TeacherAction
from server.tasks import get_task_names

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Tasks: {get_task_names()}')

## 4. WandB Login (optional)

In [ ]:
import wandb
wandb.login()
wandb.init(project='teacher-workspace-grpo', name='llama-1b-grpo-run1')

## 5. Load Model (4-bit Llama-3.2-1B)

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 2048
MODEL_NAME  = 'unsloth/Llama-3.2-1B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model loaded with LoRA adapters')
model.print_trainable_parameters()

## 6. Prompt Helpers

In [ ]:
SYSTEM_PROMPT = """You are a teacher assistant controlling Google Workspace tools.

OUTPUT RULES - non-negotiable:
- Output EXACTLY ONE JSON object per response, nothing else.
- Format: {\"tool_name\": \"<n>\", \"params\": {<key>: <value>}}
- No markdown, no explanation, no prose before or after the JSON.
- ONE action per response. Never output multiple actions.

CALCULATION RULES:
- AVERAGE of n values = (v1 + v2 + v3) / n, rounded to exactly 2 decimal places.

STOPPING RULES:
- Only perform actions explicitly required by the task prompt.
- Once all sub-tasks are complete, stop.
- Never repeat a completed step.

AVAILABLE TOOLS:
Classroom: list_classrooms, get_classroom, create_classroom, create_announcement
Sheets: list_sheets, get_cells, create_sheet, update_cell, add_note, set_formula, sort_range
Gmail: list_inbox, read_mail, send_mail, star_mail, create_label, assign_label
Calendar: list_events, create_event, create_meet_event
"""

def obs_to_prompt(obs, step: int, history: List[str]) -> str:
    sheets_str = ''
    for name, data in obs.sheets.items():
        cells = data.get('cells', {})
        sheets_str += f'\n{name}:\n'
        for k, v in list(cells.items())[:30]:
            sheets_str += f'  {k}={v}\n'

    sent_str = ''
    for m in obs.sent[-10:]:
        sent_str += f"  mail_id={m['mail_id']} to={m['to']} subject={m.get('subject','')[:30]} labels={m.get('labels',[])}\n"

    students_str = '\n'.join(f"  {s['name']} email={s['email']} id={s['id']}" for s in obs.students)
    parents_str  = '\n'.join(f"  {p['name']} email={p['email']} student_id={p['student_id']}" for p in obs.parents)

    feedback = ''
    if obs.error:
        feedback = f'\n! LAST ACTION FAILED: {obs.error}\n'
    elif obs.result:
        feedback = f'\n+ Last result: {json.dumps(obs.result)[:150]}\n'

    return f"""TASK:\n{obs.task_prompt}\n\nSTEP {step}\n\nHISTORY:\n{chr(10).join(history[-15:])}\n{feedback}\nGRADEBOOK:\n{sheets_str}\nSENT:\n{sent_str}\nSTUDENTS:\n{students_str}\nPARENTS:\n{parents_str}\nLABELS: {obs.labels}\n\nRespond with ONE JSON object:"""

def parse_action(text: str):
    text = re.sub(r'```(?:json)?', '', text).strip().replace('```', '').strip()
    for pattern in [
        lambda t: json.loads(t),
        lambda t: json.loads(t[t.find('{'):t.rfind('}')+1]),
    ]:
        try:
            d = pattern(text)
            if 'tool_name' in d:
                return d
        except:
            pass
    return None

print('Prompt helpers ready')

## 7. Rollout Function (one episode — local Llama 1B)

In [ ]:
MAX_STEPS = {'setup_new_course': 15, 'grade_and_notify': 25, 'end_of_semester': 35}

def run_episode(task_name: str, model, tokenizer, temperature: float = 0.7):
    env = TeacherWorkspaceEnvironment()
    obs = env.reset(task_name=task_name)

    history = []
    pairs   = []
    max_steps = MAX_STEPS[task_name]

    FastLanguageModel.for_inference(model)

    for step in range(1, max_steps + 1):
        if obs.done:
            break

        user_prompt = obs_to_prompt(obs, step, history)
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_prompt},
        ]

        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors='pt'
        ).to('cuda')

        with torch.no_grad():
            out = model.generate(
                input_ids,
                max_new_tokens=256,
                temperature=temperature,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        raw = tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True)

        full_prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        pairs.append((full_prompt, raw))

        action_dict = parse_action(raw)
        if action_dict is None:
            history.append(f'Step {step}: parse error')
            continue

        try:
            action = TeacherAction(tool_name=action_dict['tool_name'],
                                   params=action_dict.get('params', {}))
            obs = env.step(action)
            reward = obs.reward
            history.append(
                f"Step {step}: {action_dict['tool_name']} reward={reward:+.2f}"
                f" {'OK' if obs.success else 'FAIL'}"
            )
        except Exception as e:
            history.append(f'Step {step}: exception {e}')

    return pairs, env.final_score(), env.grade()

print('Rollout function ready')

## 7b. Qwen 72B Baseline via HF Inference API

Runs `Qwen/Qwen2.5-72B-Instruct` via the HuggingFace Inference API — **no local VRAM needed**.

This cell runs **once before training**. Results are stored in `baselines` and reused in cell 10.

Includes **exponential backoff retry** (up to 5 attempts) to handle HF rate limiting gracefully:
- Regular errors: waits 2s, 4s, 8s, 16s between retries
- Rate limit (429) errors: doubles the wait time
- Aborts episode early if 3+ consecutive failures occur

In [ ]:
from huggingface_hub import InferenceClient

# ── Config ─────────────────────────────────────────────────────────────────
HF_TOKEN                   = 'hf_YOUR_TOKEN_HERE'  # <- paste your HF token here
BASELINE_MODEL             = 'Qwen/Qwen2.5-72B-Instruct'
BASELINE_EPISODES_PER_TASK = 3    # 3 episodes x 3 tasks = 9 total
BASELINE_TEMPERATURE       = 0.1  # low temp for deterministic baseline
MAX_RETRIES                = 5    # retry attempts per API call

client = InferenceClient(token=HF_TOKEN)


def call_api_with_retry(messages: list, temperature: float, step: int):
    """
    Call the HF Inference API with exponential backoff retry.
    Returns the response text, or None if all attempts fail.
    Handles both rate limit errors (429) and transient failures.
    """
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat_completion(
                model=BASELINE_MODEL,
                messages=messages,
                max_tokens=256,
                temperature=temperature,
            )
            return response.choices[0].message.content  # success

        except Exception as e:
            error_str = str(e).lower()
            is_rate_limit = '429' in error_str or 'rate limit' in error_str or 'too many' in error_str

            if attempt == MAX_RETRIES - 1:
                print(f'    Step {step}: API failed after {MAX_RETRIES} attempts -- {e}')
                return None

            # Exponential backoff: 2s, 4s, 8s, 16s
            # Rate limit errors get 2x longer waits
            wait = (2 ** (attempt + 1)) * (2 if is_rate_limit else 1)
            reason = 'rate limited' if is_rate_limit else 'error'
            print(f'    Step {step}: API {reason} (attempt {attempt+1}/{MAX_RETRIES}), '
                  f'retrying in {wait}s -- {e}')
            time.sleep(wait)

    return None


def run_episode_api(task_name: str, temperature: float = 0.1):
    """
    Run one episode using Qwen 72B via HF Inference API.
    Returns (final_score, grade_score).
    """
    env = TeacherWorkspaceEnvironment()
    obs = env.reset(task_name=task_name)

    history    = []
    max_steps  = MAX_STEPS[task_name]
    api_errors = 0  # consecutive failure counter

    for step in range(1, max_steps + 1):
        if obs.done:
            break

        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': obs_to_prompt(obs, step, history)},
        ]

        raw = call_api_with_retry(messages, temperature, step)

        if raw is None:
            api_errors += 1
            history.append(f'Step {step}: skipped (API unavailable)')
            if api_errors >= 3:
                print(f'    Warning: 3 consecutive API failures, ending episode early')
                break
            continue

        api_errors = 0  # reset on success

        action_dict = parse_action(raw)
        if action_dict is None:
            history.append(f'Step {step}: parse error')
            continue

        try:
            action = TeacherAction(tool_name=action_dict['tool_name'],
                                   params=action_dict.get('params', {}))
            obs = env.step(action)
            history.append(
                f"Step {step}: {action_dict['tool_name']} reward={obs.reward:+.2f}"
                f" {'OK' if obs.success else 'FAIL'}"
            )
        except Exception as e:
            history.append(f'Step {step}: exception {e}')

    return env.final_score(), env.grade()


# ── Run baseline evaluation (ONE TIME before training) ─────────────────────
TASK_NAMES = get_task_names()
baselines  = {}  # populated here, read in cell 10

print(f'Running Qwen 72B baseline via HF Inference API')
print(f'Model    : {BASELINE_MODEL}')
print(f'Episodes : {BASELINE_EPISODES_PER_TASK} per task ({BASELINE_EPISODES_PER_TASK * len(TASK_NAMES)} total)')
print(f'Retries  : up to {MAX_RETRIES} attempts per call (exponential backoff)')
print('-' * 60)

for task in TASK_NAMES:
    grades = []
    print(f'\nTask: {task}')
    for ep in range(BASELINE_EPISODES_PER_TASK):
        fscore, gscore = run_episode_api(task, temperature=BASELINE_TEMPERATURE)
        grades.append(gscore)
        print(f'  Episode {ep+1}/{BASELINE_EPISODES_PER_TASK} -> grade={gscore:.3f}')
    baselines[task] = sum(grades) / len(grades)
    print(f'  Baseline avg = {baselines[task]:.3f}')

print('\n' + '-' * 60)
print('Baseline evaluation complete!')
print('baselines =', baselines)

## 8. GRPO Training Loop

In [ ]:
from torch.optim import AdamW
from unsloth import FastLanguageModel

TRAIN_STEPS       = 200
EPISODES_PER_STEP = 4
LR                = 5e-5
CLIP_EPS          = 0.2
KL_COEF           = 0.01
TASK_NAMES        = get_task_names()

optimizer = AdamW(model.parameters(), lr=LR)

reward_log = []
grade_log  = []
loss_log   = []
task_log   = []

print(f'Starting GRPO training: {TRAIN_STEPS} steps, {EPISODES_PER_STEP} episodes/step')
print(f'Tasks: {TASK_NAMES}')
print('-' * 60)

for step in range(TRAIN_STEPS):
    task_name = TASK_NAMES[step % len(TASK_NAMES)]
    task_log.append(task_name)

    all_pairs = []
    scores    = []
    grades    = []

    for _ in range(EPISODES_PER_STEP):
        pairs, fscore, gscore = run_episode(task_name, model, tokenizer, temperature=0.8)
        all_pairs.append(pairs)
        scores.append(fscore)
        grades.append(gscore)

    scores_t   = torch.tensor(scores, dtype=torch.float32)
    advantages = (scores_t - scores_t.mean()) / (scores_t.std() + 1e-8)

    FastLanguageModel.for_training(model)
    optimizer.zero_grad()

    total_loss = torch.tensor(0.0, requires_grad=True, device='cuda')

    for ep_idx, (pairs, adv) in enumerate(zip(all_pairs, advantages)):
        if not pairs:
            continue
        for (prompt_text, response_text) in pairs:
            full_text  = prompt_text + response_text
            enc        = tokenizer(full_text, return_tensors='pt',
                                   truncation=True, max_length=MAX_SEQ_LEN).to('cuda')
            prompt_enc = tokenizer(prompt_text, return_tensors='pt',
                                   truncation=True, max_length=MAX_SEQ_LEN).to('cuda')
            prompt_len = prompt_enc['input_ids'].shape[1]

            out       = model(**enc, labels=enc['input_ids'])
            logits    = out.logits[:, prompt_len-1:-1, :]
            labels    = enc['input_ids'][:, prompt_len:]
            log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
            token_lp  = log_probs.gather(2, labels.unsqueeze(-1)).squeeze(-1)
            seq_lp    = token_lp.mean()
            total_loss = total_loss + (-adv.to('cuda') * seq_lp)

    total_loss = total_loss / (EPISODES_PER_STEP + 1e-8)
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    mean_score = float(scores_t.mean())
    mean_grade = sum(grades) / len(grades)
    loss_val   = float(total_loss.detach())

    reward_log.append(mean_score)
    grade_log.append(mean_grade)
    loss_log.append(loss_val)

    if step % 10 == 0:
        print(f'Step {step:3d} | task={task_name:<20} | '
              f'grade={mean_grade:.3f} | score={mean_score:.3f} | loss={loss_val:.4f}')

    try:
        wandb.log({'step': step, 'mean_grade': mean_grade,
                   'mean_final_score': mean_score, 'loss': loss_val, 'task': task_name})
    except:
        pass

print('\nTraining complete!')

## 9. Plot Reward Curve

In [ ]:
import numpy as np

def moving_avg(arr, w=10):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(grade_log, alpha=0.3, color='steelblue', label='raw')
axes[0].plot(moving_avg(grade_log), color='steelblue', linewidth=2, label='moving avg')
axes[0].set_title('Grade Score (env.grade())', fontsize=12)
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Score (0-1)')
axes[0].set_ylim(0, 1.05)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(reward_log, alpha=0.3, color='darkorange', label='raw')
axes[1].plot(moving_avg(reward_log), color='darkorange', linewidth=2, label='moving avg')
axes[1].set_title('Final Score (grade + efficiency)', fontsize=12)
axes[1].set_xlabel('Training Step')
axes[1].set_ylabel('Score (0-1)')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(loss_log, alpha=0.3, color='crimson', label='raw')
axes[2].plot(moving_avg(loss_log), color='crimson', linewidth=2, label='moving avg')
axes[2].set_title('GRPO Loss', fontsize=12)
axes[2].set_xlabel('Training Step')
axes[2].set_ylabel('Loss')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Teacher Workspace Agent - GRPO Training (Llama-3.2-1B)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved reward_curve.png')

## 10. Before vs After Evaluation

Baselines come from cell 7b (real Qwen 72B runs via HF Inference API), not hardcoded values.

In [ ]:
assert baselines, 'baselines dict is empty -- did cell 7b run successfully?'

print('=== POST-TRAINING EVALUATION ===')
print('Running 3 episodes per task (trained Llama 1B)...')
print()

results = {}
for task in TASK_NAMES:
    task_grades = []
    for _ in range(3):
        _, _, g = run_episode(task, model, tokenizer, temperature=0.1)
        task_grades.append(g)
    results[task] = sum(task_grades) / len(task_grades)
    print(f'{task:<25} avg_grade={results[task]:.3f}')

print()
print(f'{"Task":<25} {"Baseline (Qwen72B)":>20} {"Trained (Llama1B)":>20} {"Change":>10}')
print('-' * 80)
for task in TASK_NAMES:
    b = baselines[task]
    t = results[task]
    indicator = 'UP' if t >= b else 'DOWN'
    print(f'{task:<25} {b:>20.3f} {t:>20.3f} {t-b:>+10.3f}  [{indicator}]')

avg_b = sum(baselines.values()) / len(baselines)
avg_t = sum(results.values()) / len(results)
print('-' * 80)
print(f'{"Average":<25} {avg_b:>20.3f} {avg_t:>20.3f} {avg_t-avg_b:>+10.3f}')
print()
print(f'Baseline model : {BASELINE_MODEL}')
print(f'Baseline eps   : {BASELINE_EPISODES_PER_TASK} per task')

## 11. Save & Push to HuggingFace Hub

In [ ]:
from huggingface_hub import login

login()

model.save_pretrained('teacher-agent-lora')
tokenizer.save_pretrained('teacher-agent-lora')
print('Saved locally to teacher-agent-lora/')

model.push_to_hub('amydosomething/teacher-workspace-agent-lora')
tokenizer.push_to_hub('amydosomething/teacher-workspace-agent-lora')
print('Pushed to HuggingFace Hub: amydosomething/teacher-workspace-agent-lora')

## 12. Save reward curve to repo

In [ ]:
!cp reward_curve.png /content/teacher-workspace/reward_curve.png
%cd /content/teacher-workspace
!git config user.email 'you@example.com'
!git config user.name 'amydosomething'
!git add reward_curve.png
!git commit -m 'Add GRPO training reward curve'
# Uncomment and fill in your token to push:
# !git push https://<YOUR_GITHUB_TOKEN>@github.com/amydosomething/teacher-workspace.git